# Tactic Mode vs Term Mode in Lean 4

## The core idea

In Lean 4's type theory, **a proof is just a value**. If you want to prove `P`, you need to construct
a term of type `P`. Types that live in `Prop` (propositions) are proved by exhibiting their inhabitants.

There are two syntactic styles for writing that value:

| | Term mode | Tactic mode |
|---|---|---|
| How | Write the proof expression directly | Issue commands that build the expression incrementally |
| Entry | No keyword needed | `by` keyword |
| Mental model | Functional programming | Backwards goal reduction |
| Feedback | Type errors only | Live goal state after each step |
| LeanDojo traces it? | ❌ No (single expression, no steps) | ✅ Yes (step-by-step) |

---

## Term mode examples

These are real declarations from `corpus.jsonl` (Mathlib source). Each proof is a single
expression — no `by`, no steps:

```lean
-- Proof by direct function application
lemma Equivalence.reflexive {r : β → β → Prop} (h : Equivalence r) : Reflexive r :=
  h.refl
--            ^^^^^^ the proof IS the term h.refl — the .refl field of the Equivalence structure

-- Proof by lambda + field access
lemma Equivalence.symmetric {r : β → β → Prop} (h : Equivalence r) : Symmetric r :=
  fun _ _ ↦ h.symm
--          ^^^^^^^^^^^ a lambda: given any two elements, apply h.symm

-- Proof by propext + iff construction
lemma imp_congr_eq {a b c d : Prop} (h₁ : a = c) (h₂ : b = d) : (a → b) = (c → d) :=
  propext (imp_congr h₁.to_iff h₂.to_iff)
--         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^ propext turns an Iff into an equality of Props

-- One-liner by definitional equality
lemma flip_def {f : α → β → φ} : flip f = fun b a => f a b := rfl
--                                                              ^^^ both sides reduce to the same normal form
```

---

## Tactic mode examples

```lean
-- Same proof, tactic style:
lemma Equivalence.reflexive {r : β → β → Prop} (h : Equivalence r) : Reflexive r := by
  exact h.refl  -- 'exact' says: this term has exactly the goal's type, done.

-- A more involved tactic proof:
theorem Nat.add_comm (n m : Nat) : n + m = m + n := by
  induction n with
  | zero      => simp             -- base case: 0 + m = m + 0, solved by simp
  | succ n ih => simp [Nat.succ_add, ih]  -- ih : n + m = m + n
```

In tactic mode, after each line the **proof state** changes. LeanDojo captures these states:

```
Initial goal:
  n m : Nat
  ⊢ n + m = m + n      ← the ⊢ symbol means "prove this"

After `induction n`:
  case zero
  m : Nat
  ⊢ 0 + m = m + 0

  case succ
  n ih : Nat
  ih : n + m = m + n   ← induction hypothesis, now a hypothesis
  ⊢ n + 1 + m = m + (n + 1)

After `simp` on the zero case:
  no goals             ← case closed
```

---

## Why term-mode proofs can't be traced

LeanDojo works by injecting a tracing callback at each **tactic execution boundary**.
A term-mode proof like `h.refl` is a single expression — the Lean elaborator processes it
atomically and produces the proof term. There's no step boundary to hook into.

To get sub-expression-level provenance for term-mode proofs you'd need to instrument
Lean's elaborator itself (much harder). This is why ~50% of our theorems have 0 traced tactics.

In [2]:
import json
from pathlib import Path
import duckdb

ROOT = Path().resolve().parent
db = duckdb.connect(str(ROOT / 'data' / 'mathlib.db'), read_only=True)

CORPUS = ROOT / 'data/raw/leandojo/corpus.jsonl'
TRAIN  = ROOT / 'data/raw/leandojo/leandojo_benchmark_4/random/train.json'

## How many theorems are tactic-mode vs term-mode?

In [3]:
# In our DB: theorems with at least one tactic step vs zero
db.sql("""
WITH theorem_tactic_counts AS (
    SELECT
        d.name,
        COUNT(t.tactic_idx) AS n_tactics
    FROM declarations d
    LEFT JOIN tactics t ON d.name = t.theorem_name
    WHERE d.in_leandojo
    GROUP BY d.name
)
SELECT
    CASE WHEN n_tactics > 0 THEN 'tactic-mode (traced)' ELSE 'term-mode (0 tactics)' END AS proof_style,
    COUNT(*) AS n_theorems,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM theorem_tactic_counts
GROUP BY 1
ORDER BY 2 DESC
""").show()

┌───────────────────────┬────────────┬────────┐
│      proof_style      │ n_theorems │  pct   │
│        varchar        │   int64    │ double │
├───────────────────────┼────────────┼────────┤
│ term-mode (0 tactics) │     124421 │   66.9 │
│ tactic-mode (traced)  │      61542 │   33.1 │
└───────────────────────┴────────────┴────────┘



In [4]:
# Distribution of proof lengths (number of tactic steps)
db.sql("""
SELECT
    n_tactics,
    COUNT(*) AS n_theorems
FROM (
    SELECT theorem_name, COUNT(*) AS n_tactics
    FROM tactics
    GROUP BY theorem_name
)
GROUP BY n_tactics
ORDER BY n_tactics
LIMIT 20
""").show()

┌───────────┬────────────┐
│ n_tactics │ n_theorems │
│   int64   │   int64    │
├───────────┼────────────┤
│         1 │      21497 │
│         2 │      13273 │
│         3 │       6795 │
│         4 │       4200 │
│         5 │       3086 │
│         6 │       2261 │
│         7 │       1727 │
│         8 │       1396 │
│         9 │       1198 │
│        10 │        882 │
│        11 │        720 │
│        12 │        614 │
│        13 │        480 │
│        14 │        409 │
│        15 │        356 │
│        16 │        314 │
│        17 │        249 │
│        18 │        224 │
│        19 │        202 │
│        20 │        168 │
├───────────┴────────────┤
│ 20 rows      2 columns │
└────────────────────────┘



## Real term-mode proofs from corpus.jsonl

These are short Mathlib theorems whose code has `:=` but no `by` — they write the proof value directly.

In [5]:
import re

term_mode_examples = []

with open(CORPUS) as f:
    for line in f:
        entry = json.loads(line)
        if 'Mathlib/' not in entry.get('path', ''):
            continue
        for p in entry.get('premises', []):
            code = p.get('code', '')
            name = p.get('full_name', '')
            # Short theorem/lemma with := but no tactic 'by'
            if (len(code) < 250
                    and re.match(r'(theorem|lemma)', code.strip())
                    and ':=' in code
                    and ' by' not in code
                    and '\nby' not in code
                    and len(code.split('\n')) <= 5):
                term_mode_examples.append((name, code))
                if len(term_mode_examples) >= 6:
                    break
        if len(term_mode_examples) >= 6:
            break

for name, code in term_mode_examples:
    print(f'--- {name} ---')
    print(code)
    print()

--- heq_of_eq_rec_left ---
theorem heq_of_eq_rec_left {φ : α → Sort v} {a a' : α} {p₁ : φ a} {p₂ : φ a'} :
    (e : a = a') → (h₂ : Eq.rec (motive := fun a _ ↦ φ a) p₁ e = p₂) → HEq p₁ p₂

--- heq_of_eq_rec_right ---
theorem heq_of_eq_rec_right {φ : α → Sort v} {a a' : α} {p₁ : φ a} {p₂ : φ a'} :
    (e : a' = a) → (h₂ : p₁ = Eq.rec (motive := fun a _ ↦ φ a) p₂ e) → HEq p₁ p₂

--- let_value_eq ---
theorem let_value_eq {α : Sort u} {β : Sort v} {a₁ a₂ : α} (b : α → β)
    (h : a₁ = a₂) : (let x : α := a₁; b x) = (let x : α := a₂; b x)

--- let_value_heq ---
theorem let_value_heq {α : Sort v} {β : α → Sort u} {a₁ a₂ : α} (b : ∀ x : α, β x)
    (h : a₁ = a₂) : HEq (let x : α := a₁; b x) (let x : α := a₂; b x)

--- let_body_eq ---
theorem let_body_eq {α : Sort v} {β : α → Sort u} (a : α) {b₁ b₂ : ∀ x : α, β x}
    (h : ∀ x, b₁ x = b₂ x) : (let x : α := a; b₁ x) = (let x : α := a; b₂ x)

--- let_eq ---
theorem let_eq {α : Sort v} {β : Sort u} {a₁ a₂ : α} {b₁ b₂ : α → β}
    (h₁ : a₁ = a₂) (

## A complete tactic-mode proof trace

For tactic proofs, we have the full step-by-step trace: goal before, tactic applied, goal after, premises used.

In [6]:
def show_proof_trace(theorem_name: str):
    """Print the full tactic trace for a theorem from the DB."""
    steps = db.execute("""
        SELECT t.tactic_idx, t.tactic_text, t.state_before, t.state_after,
               list(td.premise_name) AS premises
        FROM tactics t
        LEFT JOIN tactic_deps td
            ON t.theorem_name = td.theorem_name AND t.tactic_idx = td.tactic_idx
        WHERE t.theorem_name = ?
        GROUP BY t.tactic_idx, t.tactic_text, t.state_before, t.state_after
        ORDER BY t.tactic_idx
    """, [theorem_name]).fetchall()

    if not steps:
        print(f"No tactic trace found for '{theorem_name}'")
        return

    # Show type signature from declarations
    row = db.execute(
        "SELECT module, kind, type_sig FROM declarations WHERE name = ?",
        [theorem_name]
    ).fetchone()
    if row:
        print(f"{'='*60}")
        print(f"THEOREM: {theorem_name}")
        print(f"  module : {row[0]}")
        print(f"  kind   : {row[1]}")
        if row[2]:
            print(f"  type   : {row[2][:120]}")
        print(f"{'='*60}")

    # Print initial goal
    print(f"\nINITIAL GOAL:")
    print(steps[0][2])  # state_before of step 0

    for idx, tactic_text, state_before, state_after, premises in steps:
        print(f"\n{'─'*50}")
        print(f"Step {idx}: {tactic_text}")
        if premises:
            print(f"  Uses  : {', '.join(p for p in premises if p)}")
        print(f"  After : {state_after}")

In [7]:
# Find a short, readable tactic proof to display
# (2-4 steps, small state, has some premises)
candidates = db.execute("""
    SELECT t.theorem_name, COUNT(*) AS n_steps, SUM(LENGTH(t.state_before)) AS state_size
    FROM tactics t
    JOIN tactic_deps td ON t.theorem_name = td.theorem_name
    JOIN declarations d  ON t.theorem_name = d.name
    WHERE d.module LIKE 'Mathlib%'
    GROUP BY t.theorem_name
    HAVING n_steps BETWEEN 5 AND 7
       AND state_size < 1500
    ORDER BY state_size
    LIMIT 5
""").fetchall()

print("Good candidates for display:")
for name, n, sz in candidates:
    print(f"  {name}  ({n} steps, state_size={sz})")

Good candidates for display:
  Real.Angle.pi_ne_zero  (6 steps, state_size=45)
  mk_complex  (6 steps, state_size=48)
  EReal.inv_zero  (6 steps, state_size=57)
  inv_goldConj  (6 steps, state_size=60)
  Cardinal.continuum_power_aleph0  (5 steps, state_size=60)


In [8]:
# Show the full trace for the first candidate (or pick any name)
if candidates:
    show_proof_trace(candidates[0][0])

THEOREM: Real.Angle.pi_ne_zero
  module : Mathlib.Analysis.SpecialFunctions.Trigonometric.Angle
  kind   : theorem
  type   : ↑Real.pi ≠ 0

INITIAL GOAL:
⊢ ↑π ≠ 0

──────────────────────────────────────────────────
Step 0: rw [← toReal_injective.ne_iff, toReal_pi, toReal_zero]
  Uses  : Real.Angle.toReal_pi, Real.Angle.toReal_zero
  After : ⊢ π ≠ 0

──────────────────────────────────────────────────
Step 1: exact Real.pi_ne_zero
  Uses  : Real.pi_ne_zero
  After : no goals


In [9]:
# You can look up any theorem by name, e.g.:
# show_proof_trace('Finset.sum_comm')
# show_proof_trace('List.length_append')

## Tactic-level dependency graph for one theorem

The `tactic_deps` table tells us *which step* uses which premise — finer than the theorem-level `dependencies` table.

In [10]:
# Compare: theorem-level deps vs tactic-level deps for a specific theorem
example = candidates[0][0] if candidates else None

if example:
    print(f"Theorem: {example}")
    print()

    print("Theorem-level edges (dependencies table — flat set):")
    db.sql(f"""
        SELECT dst FROM dependencies WHERE src = '{example}' ORDER BY dst
    """).show()

    print("Tactic-level edges (tactic_deps table — which step uses which):")
    db.sql(f"""
        SELECT tactic_idx, premise_name
        FROM tactic_deps
        WHERE theorem_name = '{example}'
        ORDER BY tactic_idx, premise_name
    """).show()

Theorem: Real.Angle.pi_ne_zero

Theorem-level edges (dependencies table — flat set):
┌────────────────────────┐
│          dst           │
│        varchar         │
├────────────────────────┤
│ Real.Angle.toReal_pi   │
│ Real.Angle.toReal_zero │
│ Real.pi_ne_zero        │
└────────────────────────┘

Tactic-level edges (tactic_deps table — which step uses which):
┌────────────┬────────────────────────┐
│ tactic_idx │      premise_name      │
│   int32    │        varchar         │
├────────────┼────────────────────────┤
│          0 │ Real.Angle.toReal_pi   │
│          0 │ Real.Angle.toReal_zero │
│          1 │ Real.pi_ne_zero        │
└────────────┴────────────────────────┘



## Most-used premises per tactic keyword

Which lemmas are most often invoked by `simp` vs `rw` vs `apply`?

In [11]:
db.sql("""
WITH tactic_kw AS (
    SELECT
        regexp_extract(tactic_text, '^(\\w+)', 1) AS keyword,
        theorem_name,
        tactic_idx
    FROM tactics
    WHERE tactic_text IS NOT NULL
)
SELECT
    kw.keyword,
    td.premise_name,
    COUNT(*) AS uses
FROM tactic_kw kw
JOIN tactic_deps td
    ON kw.theorem_name = td.theorem_name AND kw.tactic_idx = td.tactic_idx
WHERE kw.keyword IN ('rw', 'simp', 'apply', 'exact', 'have')
GROUP BY kw.keyword, td.premise_name
QUALIFY ROW_NUMBER() OVER (PARTITION BY kw.keyword ORDER BY uses DESC) <= 5
ORDER BY kw.keyword, uses DESC
""").show(max_rows=30)

┌─────────┬───────────────────────────────┬───────┐
│ keyword │         premise_name          │ uses  │
│ varchar │            varchar            │ int64 │
├─────────┼───────────────────────────────┼───────┤
│ apply   │ le_antisymm                   │   301 │
│ apply   │ rfl                           │   114 │
│ apply   │ LE.le.trans                   │    95 │
│ apply   │ le_trans                      │    89 │
│ apply   │ congr_arg                     │    76 │
│ exact   │ rfl                           │  1398 │
│ exact   │ Eq.symm                       │   765 │
│ exact   │ False.elim                    │   364 │
│ exact   │ Or.inl                        │   352 │
│ exact   │ Or.inr                        │   314 │
│ have    │ rfl                           │   459 │
│ have    │ Filter.Tendsto                │   338 │
│ have    │ Set                           │   292 │
│ have    │ Eq.symm                       │   265 │
│ have    │ Filter.atTop                  │   255 │
│ rw      │ 

## Proof complexity by module area

Do some areas of Mathlib have longer, more tactic-heavy proofs on average?

In [12]:
db.sql("""
WITH theorem_lengths AS (
    SELECT t.theorem_name, COUNT(*) AS n_steps
    FROM tactics t
    GROUP BY t.theorem_name
)
SELECT
    d.module_parts[2] AS area,
    COUNT(*)          AS n_theorems,
    ROUND(AVG(tl.n_steps), 2) AS avg_steps,
    MAX(tl.n_steps)   AS max_steps
FROM declarations d
JOIN theorem_lengths tl ON d.name = tl.theorem_name
WHERE d.module_parts[1] = 'Mathlib'
  AND d.module_parts[2] IS NOT NULL
GROUP BY area
HAVING n_theorems >= 50
ORDER BY avg_steps DESC
LIMIT 15
""").show()

┌───────────────────┬────────────┬───────────┬───────────┐
│       area        │ n_theorems │ avg_steps │ max_steps │
│      varchar      │   int64    │  double   │   int64   │
├───────────────────┼────────────┼───────────┼───────────┤
│ Computability     │        561 │      7.78 │       126 │
│ NumberTheory      │       2135 │      6.47 │       100 │
│ MeasureTheory     │       4166 │      5.98 │       122 │
│ AlgebraicTopology │        231 │      5.85 │        57 │
│ Probability       │       1077 │      5.81 │        88 │
│ RingTheory        │       3539 │      5.51 │        84 │
│ FieldTheory       │        751 │      4.97 │        55 │
│ Analysis          │       7143 │      4.87 │       128 │
│ Geometry          │       1201 │      4.76 │       123 │
│ Combinatorics     │       1551 │      4.69 │        72 │
│ ModelTheory       │        309 │      4.65 │        43 │
│ GroupTheory       │       1607 │      4.62 │        58 │
│ AlgebraicGeometry │        826 │      4.37 │        58